In [ ]:
import torch

from aare.AareDataset import AareDataset
from aare.params import read_params
from aare.preparation import (
    resample,
    remove_faulty_periods,
    remove_outliers,
    interpolate,
)
from aare.remote_existenz_store import RemoteExistenzStore

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset.from_conf()

In [ ]:
df = ds.get_val()
o_df = df.copy()
df

In [ ]:
df = resample(df)
df = remove_faulty_periods(df)
df = remove_outliers(df)
df = interpolate(df, drop_filled=True)

# ARIMA(X) (with just air temp)

Reasons I think this might work:

1. The water temperatures seems to follow an AR(1) process and is stationary after first difference
2. The air temperature (forecast), which is probably our most dominant covariate/predictor, captures the same seasonality as the water temperature (daily and yearly), so we don't need to worry about seasonality. This is an especially important point because ARIMA cannot handle multiple seasonalities (should use TBATS instead for example).

Reasons this might not perform very well:

1. The relationship between water and air temperature is non-linear at low and high temperatures (DOI 10.1029/98WR01877). Could use some non-linear transformations of the air temp as covariates to help with this.
2. We already know lag 1 (hour) of the air temperature is the best lag (highest correlation), so there is a seasonality discrepancy of 1 hour
3. There are other factors that influence the water temperature and the relationships and interactions are probably much more complex than ARIMAX can model.
